# 03 — Classifier Training, Validation & Calibration

Trains three models: LogReg (baseline), XGBoost (primary), RandomForest (secondary).
Selects best by val F1. Applies Platt scaling. Saves winning model.

**Ethical constraint:** Output is `support_recommendation_score` 0–100. Never a clinical label.

In [ ]:
import sys
from pathlib import Path

repo_root = Path(".").resolve().parent
sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    classification_report,
    precision_recall_curve,
    roc_curve,
    RocCurveDisplay,
)

from src.risk_classifier import (
    FEATURE_COLS, TARGET_COL, SCORE_BANDS,
    train, evaluate, save_model, predict_score,
)

df = pd.read_csv(repo_root / "data" / "synthetic" / "student_wellbeing.csv")
print(f"Dataset: {df.shape} | prevalence: {df[TARGET_COL].mean():.4f}")

## 1. Train All Three Models

In [ ]:
bundles = {}
for model_name in ["logistic_regression", "xgboost", "random_forest"]:
    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")
    bundles[model_name] = train(
        df,
        model_name=model_name,
        random_state=42,
        use_smote=True,
        cv_folds=5,
        tune_hyperparams=True,
        verbose=True,
    )

## 2. Model Comparison

In [ ]:
comparison = []
for name, bundle in bundles.items():
    comparison.append({
        "model": name,
        "cv_f1": f"{bundle.cv_f1_mean:.4f} ± {bundle.cv_f1_std:.4f}",
        "val_f1": f"{bundle.val_metrics['f1']:.4f}",
        "val_auc": f"{bundle.val_metrics['roc_auc']:.4f}",
        "val_brier": f"{bundle.val_metrics['brier_score']:.4f}",
        "test_f1": f"{bundle.train_metrics['f1']:.4f}",
        "test_auc": f"{bundle.train_metrics['roc_auc']:.4f}",
    })

comparison_df = pd.DataFrame(comparison).set_index("model")
print(comparison_df.to_string())

# Pick primary model: XGBoost (per plan spec)
primary = bundles["xgboost"]
print(f"\nPrimary model: XGBoost (per Phase 2 spec)")

## 3. ROC and Precision-Recall Curves

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Recreate test split (same seed = same split)
X = df[FEATURE_COLS].values
y = df[TARGET_COL].values
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
_, test_idx = next(sss.split(X, y))
X_test, y_test = X[test_idx], y[test_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = {"logistic_regression": "steelblue", "xgboost": "tomato", "random_forest": "seagreen"}

for name, bundle in bundles.items():
    probs = bundle.calibrated_pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = bundle.train_metrics["roc_auc"]
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=colors[name])

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve (test set)")
axes[0].legend()

for name, bundle in bundles.items():
    probs = bundle.calibrated_pipeline.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap = bundle.train_metrics["avg_precision"]
    axes[1].plot(rec, prec, label=f"{name} (AP={ap:.3f})", color=colors[name])

axes[1].axhline(y_test.mean(), color="gray", linestyle="--", label=f"Baseline ({y_test.mean():.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve (test set)")
axes[1].legend()

plt.tight_layout()
plt.savefig(repo_root / "notebooks" / "roc_pr_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Calibration Plot (XGBoost primary)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

for name, bundle in bundles.items():
    probs = bundle.calibrated_pipeline.predict_proba(X_test)[:, 1]
    prob_true, prob_pred = calibration_curve(y_test, probs, n_bins=10)
    ax.plot(prob_pred, prob_true, marker="o", label=name, color=colors[name])

ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration curves (Platt scaling applied)")
ax.legend()
plt.tight_layout()
plt.savefig(repo_root / "notebooks" / "calibration_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"XGBoost Brier score: {primary.train_metrics['brier_score']:.4f} (lower = better calibration)")

## 5. Score Band Distribution

In [ ]:
probs_all = primary.calibrated_pipeline.predict_proba(X)[:, 1]
scores_all = (probs_all * 100).clip(0, 100)

band_counts = {"baseline": 0, "check_in_suggested": 0, "outreach_recommended": 0, "priority_follow_up": 0}
from src.risk_classifier import score_to_band
for s in scores_all:
    band_counts[score_to_band(s)] += 1

print("Score band distribution (full dataset):")
for band, count in band_counts.items():
    pct = count / len(scores_all) * 100
    print(f"  {band}: {count:,} ({pct:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores_all[y == 0], bins=50, alpha=0.6, label="No recommendation (actual)", color="steelblue", density=True)
ax.hist(scores_all[y == 1], bins=50, alpha=0.6, label="Recommended (actual)", color="tomato", density=True)
for band, (lo, hi) in SCORE_BANDS.items():
    ax.axvline(lo, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Support recommendation score")
ax.set_ylabel("Density")
ax.set_title("Score distribution by actual label")
ax.legend()
plt.tight_layout()
plt.savefig(repo_root / "notebooks" / "score_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Example Predictions (Counselor-Facing Output)

In [ ]:
# Show 5 high-scoring predictions with their explanations
sample_df = df[FEATURE_COLS].iloc[test_idx]
results = predict_score(primary, sample_df, compute_shap=True)

high_risk = sorted(results, key=lambda r: r.score, reverse=True)[:5]
print("=== Top 5 high-score predictions (counselor-facing output) ===\n")
for r in high_risk:
    print(r.display_text)
    print()

## 7. Save Primary Model

In [ ]:
model_path = save_model(primary)
print(f"Model saved to: {model_path}")
print(f"Test F1: {primary.train_metrics['f1']:.4f}")
print(f"Test AUC: {primary.train_metrics['roc_auc']:.4f}")
print(f"Test Brier: {primary.train_metrics['brier_score']:.4f}")